In [18]:
import pandas as pd
import numpy as np
import joblib
import httpagentparser
import json
import re
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
import torch
import torch.nn as nn
from app.models.log import HttpRequestLog  # Import your HttpRequestLog model
from joblib import load


In [11]:

def get_log_data(log):
    re_exp = '(^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+) ([0-9]*) \"(.*?) (.*?)\" (.*?) (.*?) \"(.*?)\" \"(.*?)\" \"(.*?)\" \"(.*?)\" (.*?) ({.*?}) ({.*?}) ({.*?})'
    # print(log,"log")
    match = re.search(re_exp, log)
    if not match:
        print("NO MATCH FOUNND")
        return None

    return {
        'client_host': match.group(1),
        'request_time': float(match.group(2)) if match.group(2) else None,
        'request_method': match.group(3),
        'request_path': match.group(4),
        'http_version': match.group(5),
        'response_status': int(match.group(6).split()[0]) if match.group(6).split() else None,
        'process_time': float(match.group(6).split()[1]) if len(match.group(6).split()) > 1 else None,
        'referrer': match.group(7),
        'user_agent': match.group(8),
        'user_id': int(match.group(10)) if match.group(10).isdigit() else None,
        'log_message': match.group(11),
        'cookies': match.group(12),
        'post_params': match.group(13),
        'get_params': match.group(14),
        'body': None
    }

def create_http_request_log_instance_from_string(log_string):
    log_data = get_log_data(log_string)
    if log_data is None:
        return None  # Or handle the error as per your application's need
    # print(log_data,111)
    # Create an instance of HttpRequestLog with the extracted data
    log_instance = HttpRequestLog(
        client_host=log_data['client_host'],
        request_time=log_data['request_time'],
        request_method=log_data['request_method'],
        request_path=log_data['request_path'],
        http_version=log_data['http_version'],
        response_status=log_data['response_status'],
        process_time=log_data['process_time'],
        referrer=log_data['referrer'],
        user_agent=log_data['user_agent'],
        user_id=log_data['user_id'],
        log_message=log_data['log_message'],
        cookies=log_data['cookies'],
        post_params=log_data['post_params'],
        get_params=log_data['get_params'],
        body=log_data['body']
    )

    return log_instance


In [27]:

def inference_preprocess_data(log_instance, one_hot_encoder):
    if log_instance is None:
        return None
    
    log_data = log_instance.__dict__

    # Handle JSON fields
    json_features = ['cookies', 'post_params', 'get_params']
    for field in json_features:
        log_data[field] = json.loads(log_data[field]) if log_data[field] else {}
        log_data[f"{field}_count"] = len(log_data[field].keys())

    # Handling textual data (e.g., user_agent)
    log_data['browser'] = httpagentparser.detect(log_data['user_agent'])['browser']['name'] if httpagentparser.detect(log_data['user_agent']).get('browser') else 'Unknown'

    # Prepare DataFrame for one-hot encoding
    df = pd.DataFrame([log_data])
    df = df.drop(['_sa_instance_state'], axis=1)

    # Fill missing categorical columns with 'Unknown'
    categorical_features = ['request_method', 'http_version', 'user_id', 'browser']
    for feature in categorical_features:
        if feature not in df.columns:
            df[feature] = df[feature].fillna('Unknown')

    # Apply one-hot encoding to the categorical features
    encoded_df = pd.DataFrame(one_hot_encoder.transform(df[categorical_features].fillna('Unknown')).toarray(), columns=one_hot_encoder.get_feature_names_out())

    # Concatenate encoded features with the rest of the data
    df = pd.concat([df.drop(columns=categorical_features), encoded_df], axis=1)

    # Drop the original categorical and JSON columns as they are now encoded
    fields_to_drop = json_features + ['user_agent', 'referrer', 'log_message', 'body', 'client_host', 'request_path']
    df = df.drop(columns=fields_to_drop)
    
    return df

In [47]:
class LogAnomalyDetector:
    def __init__(self, normal_file, anomaly_file, models):
        self.normal_file = normal_file
        self.anomaly_file = anomaly_file
        self.models = models  # Dictionary of models { 'model_name': model_path }
        self.one_hot_encoder = load('models/one_hot_encoder.joblib')  # Load or define your OneHotEncoder here
        self.normal_logs = self.read_logs(normal_file)
        self.anomaly_logs = self.read_logs(anomaly_file)

        self.normal_df = self.label_data(self.preprocess_logs(self.normal_logs), 1)  # Label 0 for normal
        self.anomaly_df = self.label_data(self.preprocess_logs(self.anomaly_logs), 0)  # Label 1 for anomalous

        self.combined_df = pd.concat([self.normal_df, self.anomaly_df])

    def read_logs(self, file_path):
        with open(file_path, 'r') as file:
            logs = file.readlines()
        return [create_http_request_log_instance_from_string(log) for log in logs]

    def preprocess_logs(self, log_instances):
        processed_data = [inference_preprocess_data(instance, self.one_hot_encoder) for instance in log_instances if instance is not None]
        processed_data = [df for df in processed_data if df is not None]  # Filter out None dataframes
        return pd.concat(processed_data, ignore_index=True) if processed_data else pd.DataFrame()

    def label_data(self, df, label):
        df['label'] = label
        return df

    def load_and_evaluate_models(self, df):
        results = {}
        for model_name, model_path in self.models.items():
            print(f"Testing model: {model_name}")
            model = joblib.load(model_path)
            # Check if the model supports the predict method
            if hasattr(model, 'predict'):
                predictions = model.predict(df.drop('label', axis=1))
            else:
                # Handle models that do not support predict
                # For example, using fit_predict for LOF with novelty=False
                predictions = model.fit_predict(df.drop('label', axis=1))

                # In this case, you may need to handle it differently 
                # or retrain the model with the correct parameters
                # continue  # or handle appropriately
            
            predictions = model.predict(df.drop('label', axis=1))
            # Adjust prediction labels based on your model's specifics

            # Compute evaluation metrics
            report = classification_report(df['label'], predictions, output_dict=True)
            accuracy = accuracy_score(df['label'], predictions)
            f1 = f1_score(df['label'], predictions, average='weighted')

            results[model_name] = {
                'classification_report': report,
                'accuracy': accuracy,
                'f1_score': f1
            }
        return results

    def run(self):
        

        return self.load_and_evaluate_models(self.combined_df)

        # return combined_df

In [48]:
# Example usage
detector = LogAnomalyDetector('logs/non_anomaly.txt', 'logs/small_anomaly.txt', {
    'IsolationForest': 'models/IsolationForest_model.pkl',
    # 'LOF': 'models/lof_model.pkl',
    # 'SVM': 'models/oc_svm_model.pkl'
    # Add other models here
})


In [49]:
detector.run()

Testing model: IsolationForest


/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(res

{'IsolationForest': {'classification_report': {'-1': {'precision': 0.0,
    'recall': 0.0,
    'f1-score': 0.0,
    'support': 0},
   '0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 9293},
   '1': {'precision': 0.6391835101060637,
    'recall': 0.9004792782633211,
    'f1-score': 0.7476591760299626,
    'support': 10641},
   'accuracy': 0.4806862646734223,
   'macro avg': {'precision': 0.21306117003535455,
    'recall': 0.300159759421107,
    'f1-score': 0.24921972534332085,
    'support': 19934},
   'weighted avg': {'precision': 0.3412035582943024,
    'recall': 0.4806862646734223,
    'f1-score': 0.399109124718312,
    'support': 19934}},
  'accuracy': 0.4806862646734223,
  'f1_score': 0.399109124718312}}

In [55]:
model_name = "models/lof_model.pkl"
model = joblib.load(model_name)
results = {}
# Check if the model supports the predict method
if hasattr(model, 'predict'):
    print("using model.predicet")
    predictions = model.predict(detector.combined_df.drop('label', axis=1))
else:
    print("using model.fit_predict")
    # Handle models that do not support predict
    # For example, using fit_predict for LOF with novelty=False
    predictions = model.fit_predict(detector.combined_df.drop('label', axis=1))

    # In this case, you may need to handle it differently 
    # or retrain the model with the correct parameters
    # continue  # or handle appropriately

predictions = model.predict(detector.combined_df.drop('label', axis=1))
# Adjust prediction labels based on your model's specifics

# Compute evaluation metrics
report = classification_report(detector.combined_df['label'], predictions, output_dict=True)
accuracy = accuracy_score(detector.combined_df['label'], predictions)
f1 = f1_score(detector.combined_df['label'], predictions, average='weighted')

results[model_name] = {
    'classification_report': report,
    'accuracy': accuracy,
    'f1_score': f1
}

using model.predicet


/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter

In [56]:
results

{'models/lof_model.pkl': {'classification_report': {'-1': {'precision': 0.0,
    'recall': 0.0,
    'f1-score': 0.0,
    'support': 0},
   '0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 9293},
   '1': {'precision': 0.9867523083099157,
    'recall': 0.6929799830842965,
    'f1-score': 0.8141768797615103,
    'support': 10641},
   'accuracy': 0.3699207384368416,
   'macro avg': {'precision': 0.3289174361033052,
    'recall': 0.23099332769476552,
    'f1-score': 0.2713922932538368,
    'support': 19934},
   'weighted avg': {'precision': 0.5267398069993886,
    'recall': 0.36992073843684153,
    'f1-score': 0.43461704512602745,
    'support': 19934}},
  'accuracy': 0.3699207384368416,
  'f1_score': 0.43461704512602745}}